# Socbench Training — GPT-2 124M on Kaggle 2x T4

This notebook trains GPT-2 124M from scratch on a pre-tokenized dataset.
Data is prepared server-side and uploaded as a Kaggle dataset.

**Setup:**
1. Upload pre-tokenized `.bin` file as a Kaggle dataset
2. Reference it in `kernel-metadata.json` → `dataset_sources`
3. Set `DATASET_NAME` below to match your uploaded dataset
4. Push with `kaggle kernels push`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG — Change this to match your uploaded dataset
# ═══════════════════════════════════════════════════════════════════════════
DATASET_NAME = "socbench/YOUR_DATASET_HERE"  # e.g. "socbench/bigcode_the-stack"
BINARY_FILENAME = "train.bin"
TOKEN_BUDGET = 1_000_000_000  # 1B tokens

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB")

In [ ]:
# Verify data
safe_name = DATASET_NAME.replace("/", "_").replace("-", "_")
data_path = f"/kaggle/input/{safe_name}/{BINARY_FILENAME}"

if os.path.exists(data_path):
    size_bytes = os.path.getsize(data_path)
    size_gb = size_bytes / (1024**3)
    n_tokens = size_bytes // 2  # uint16 = 2 bytes per token
    print(f"Data: {data_path}")
    print(f"Size: {size_gb:.2f} GB")
    print(f"Tokens: {n_tokens:,}")
else:
    print(f"ERROR: Data not found at {data_path}")
    print("Available in /kaggle/input/"):
    for d in os.listdir("/kaggle/input/"):
        print(f"  {d}")
        for f in os.listdir(f"/kaggle/input/{d}"):
            print(f"    {f}")

In [ ]:
# Save training script
script = r'''
import os, sys, json, time, math, pickle
from contextlib import nullcontext
import numpy as np
import torch
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group

# Config
dataset_path = "PLACEHOLDER_DATA_PATH"
out_dir = "/kaggle/working"
eval_interval = 500
log_interval = 100
eval_iters = 200
n_layer, n_head, n_embd = 12, 12, 768
block_size, vocab_size, dropout, bias = 1024, 50257, 0.0, True
learning_rate, max_iters = 6e-4, PLACEHOLDER_MAX_ITERS
weight_decay, beta1, beta2, grad_clip = 0.1, 0.9, 0.95, 1.0
warmup_iters = PLACEHOLDER_WARMUP
lr_decay_iters = max_iters
min_lr = 6e-5

# DDP
ddp = int(os.environ.get("RANK", -1)) != -1
if ddp:
    init_process_group(backend="nccl")
    ddp_rank, ddp_local_rank = int(os.environ["RANK"]), int(os.environ["LOCAL_RANK"])
    ddp_world_size = int(os.environ["WORLD_SIZE"])
    device = f"cuda:{ddp_local_rank}"
    torch.cuda.set_device(device)
    master_process = ddp_rank == 0
    seed_offset = ddp_rank
    gradient_accumulation_steps = 32 // ddp_world_size
else:
    device, master_process, seed_offset, gradient_accumulation_steps = "cuda", True, 0, 32

if master_process:
    os.makedirs(out_dir, exist_ok=True)

torch.manual_seed(1337 + seed_offset)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
ctx = torch.amp.autocast(device_type="cuda", dtype=torch.float16)

# Data
data = np.memmap(dataset_path, dtype=np.uint16, mode="r")
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
batch_size = 16

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(d[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(d[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

# Model (GPT-2 124M)
class CausalSelfAttention(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn = torch.nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = torch.nn.Linear(config.n_embd, config.n_embd)
        self.attn_dropout = torch.nn.Dropout(config.dropout)
        self.resid_dropout = torch.nn.Dropout(config.dropout)
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.dropout = config.dropout
        self.flash = hasattr(torch.nn.functional, "scaled_dot_product_attention")

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        if self.flash:
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = torch.nn.functional.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        y = self.resid_dropout(self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C)))
        return y

class MLP(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = torch.nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = torch.nn.GELU()
        self.c_proj = torch.nn.Linear(4 * config.n_embd, config.n_embd)
        self.dropout = torch.nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = torch.nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = torch.nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        return x + self.mlp(self.ln_2(x + self.attn(self.ln_1(x))))

class GPTConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items(): setattr(self, k, v)

class GPT(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.transformer = torch.nn.ModuleDict(dict(
            wte=torch.nn.Embedding(config.vocab_size, config.n_embd),
            wpe=torch.nn.Embedding(config.block_size, config.n_embd),
            drop=torch.nn.Dropout(config.dropout),
            h=torch.nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=torch.nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = torch.nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, torch.nn.Linear):
            torch.nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, torch.nn.Embedding):
            torch.nn.init.normal_(module.weight, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h: x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1) if targets is not None else None
        return logits, loss

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        decay, nodecay = set(), set()
        for mn, m in self.named_modules():
            for pn, p in m.named_parameters():
                fpn = f"{mn}.{pn}" if mn else pn
                if pn.endswith("bias") or pn.endswith("weight") and isinstance(m, (torch.nn.LayerNorm, torch.nn.Embedding)):
                    nodecay.add(fpn)
                else:
                    decay.add(fpn)
        optim_groups = [{"params": [p for n, p in self.named_parameters() if n in decay], "weight_decay": weight_decay}, {"params": [p for n, p in self.named_parameters() if n in nodecay], "weight_decay": 0.0}]
        use_fused = device_type == "cuda" and torch.cuda.is_available()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, eps=1e-8, fused=use_fused)
        return optimizer

config = GPTConfig(n_layer=n_layer, n_head=n_head, n_embd=n_embd, block_size=block_size, vocab_size=vocab_size, dropout=dropout, bias=bias)
model = GPT(config).to(device)
num_params = sum(p.numel() for p in model.parameters())
if master_process: print(f"Parameters: {num_params:,}")

if ddp: model = DDP(model, device_ids=[ddp_local_rank])
raw_model = model.module if ddp else model
optimizer = raw_model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), "cuda")
model = torch.compile(model)

def get_lr(it):
    if it < warmup_iters: return learning_rate * it / warmup_iters
    if it > lr_decay_iters: return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) * (learning_rate - min_lr)

# Train
X, Y = get_batch("train")
t0 = time.time()
losses_log = []

for iter_num in range(max_iters):
    lr = get_lr(iter_num)
    for pg in optimizer.param_groups: pg["lr"] = lr

    if iter_num % eval_interval == 0 and master_process:
        losses = torch.zeros(eval_iters)
        model.eval()
        for k in range(eval_iters):
            x, y = get_batch("val")
            with ctx: _, loss = model(x, y)
            losses[k] = loss.item()
        model.train()
        vl = losses.mean().item()
        losses_log.append(vl)
        print(f"iter {iter_num}: val loss {vl:.4f}")

    model.train()
    optimizer.zero_grad(set_to_none=True)
    for micro_step in range(gradient_accumulation_steps):
        if ddp: model.require_backward_grad_sync = (micro_step == gradient_accumulation_steps - 1)
        with ctx:
            _, loss = model(X, Y)
            loss = loss / gradient_accumulation_steps
        loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if iter_num % log_interval == 0 and master_process:
        t1 = time.time()
        tok_s = batch_size * block_size * gradient_accumulation_steps / max(t1 - t0, 1e-6)
        print(f"iter {iter_num}: loss {loss.item():.4f}, lr {lr:.6e}, {tok_s:.0f} tok/s")
        t0 = t1

    X, Y = get_batch("train")

if master_process:
    torch.save({"model": raw_model.state_dict(), "config": vars(config), "losses_log": losses_log}, os.path.join(out_dir, "ckpt_final.pt"))
    with open(os.path.join(out_dir, "loss_curve.json"), "w") as f:
        json.dump({"losses": losses_log, "best_val_loss": min(losses_log) if losses_log else None}, f)
    print(f"Done! Best val loss: {min(losses_log):.4f}" if losses_log else "Done!")

if ddp: destroy_process_group()
'''

# Replace placeholders
data_path = f"/kaggle/input/{safe_name}/{BINARY_FILENAME}"
script = script.replace("PLACEHOLDER_DATA_PATH", data_path)

size_bytes = os.path.getsize(data_path) if os.path.exists(data_path) else 0
n_tokens = size_bytes // 2
max_iters = n_tokens // (16 * 1024 * 32)
warmup = 375_000_000 // (16 * 1024 * 32)
script = script.replace("PLACEHOLDER_MAX_ITERS", str(max_iters))
script = script.replace("PLACEHOLDER_WARMUP", str(warmup))

with open("train.py", "w") as f:
    f.write(script)
print(f"Training script ready. {max_iters} iterations, {warmup} warmup steps.")

In [ ]:
# Launch training with DDP on 2x T4
!torchrun --nproc_per_node=2 train.py

In [ ]:
# Check results
import json

loss_file = "/kaggle/working/loss_curve.json"
if os.path.exists(loss_file):
    with open(loss_file) as f:
        data = json.load(f)
    print(f"Best val loss: {data['best_val_loss']:.4f}")
    print(f"Total eval points: {len(data['losses'])}")
    print(f"Loss curve: {data['losses'][:5]}...")
else:
    print("No loss curve found")

# List outputs
print("\nOutputs:")
for f in os.listdir("/kaggle/working/"):
    size = os.path.getsize(f"/kaggle/working/{f}")
    print(f"  {f} ({size:,} bytes)")